In [4]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import logging


In [6]:
# Configuração de Logs para monitorar o processo
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def extract_data(file_path: str) -> pd.DataFrame: 
    #logging.info(f"Extraindo dados de: {file_path}")
    df = pd.read_csv(file_path, encoding='latin1')
    # Padroniza nomes: 'PRIMEIRO NOME' -> 'primeiro_nome'
    df.columns = [col.replace(' ', '_').lower() for col in df.columns]
    logging.info("Extração concluída com sucesso.")
    return df

def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    #logging.info("Iniciando transformação de dados...")
    
    # 1 Tratamento de Datas
    date_cols = ['order_date', 'ship_date']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], format='%m/%d/%Y', errors='coerce')

    # 2 Limpeza de Strings (Trim e Title Case)
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].str.strip().str.title()

    # 3 Correção específica de Burlington/Vermont e Postal Code. 
    # Formata para ter 5 dígitos preenchendo com zero à esquerda
    df['postal_code'] = df['postal_code'].fillna(5401).astype(int).astype(str).str.zfill(5)

    # 4. Criação de colunas para verificação de entregas
    df['ship_time_days'] = (df['ship_date'] - df['order_date']).dt.days
    
    # Sazonalidade
    df['order_month'] = df['order_date'].dt.month
    df['order_year'] = df['order_date'].dt.year

    logging.info("Transformação concluída com sucesso.")
    return df


def load_to_sql(df: pd.DataFrame, server: str, db_name: str, table_name: str):
    try:
        connection_string = (
            f"mssql+pyodbc://@{server}/{db_name}"
            f"?driver=ODBC+Driver+17+for+SQL+Server"
            f"&trusted_connection=yes"
        )

        engine = create_engine(connection_string)

        logging.info(f"Conectando ao banco {db_name}...")

        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='replace',
            index=False
        )

        logging.info(f"Tabela '{table_name}' criada com sucesso!")

    except Exception as e:
        logging.error(f"Erro ao carregar para o SQL Server: {e}")

def main():
    """Função principal que orquestra o fluxo."""
    # Configurações
    FILE_PATH = '../data/raw/superstore.csv'
    SERVER = 'DESKTOP-IMHTDOG'
    DB_NAME = 'superstore_analytics'
    TABLE_NAME = 'superstore_table'

    # Fluxo ETL
    try:
        data_raw = extract_data(FILE_PATH)
        data_transformed = transform_data(data_raw)
        
        # Opcional: Mostrar um resumo antes de subir
        logging.info(f"Resumo da Tabela:\n{data_transformed.head(3)}")
        
        load_to_sql(data_transformed, SERVER, DB_NAME, TABLE_NAME)
        
        logging.info("--- PROCESSO FINALIZADO COM SUCESSO ---")
        
    except FileNotFoundError:
        logging.error("Arquivo CSV não encontrado. Verifique o caminho.")
    except Exception as e:
        logging.error(f"Ocorreu um erro inesperado: {e}")

if __name__ == "__main__":
    main()

2026-04-28 11:29:26,767 - INFO - Extração concluída com sucesso.
2026-04-28 11:29:26,812 - INFO - Transformação concluída com sucesso.
2026-04-28 11:29:26,816 - INFO - Resumo da Tabela:
   row_id        order_id order_date  ship_date     ship_mode customer_id  \
0       1  Ca-2016-152156 2016-11-08 2016-11-11  Second Class    Cg-12520   
1       2  Ca-2016-152156 2016-11-08 2016-11-11  Second Class    Cg-12520   
2       3  Ca-2016-138688 2016-06-12 2016-06-16  Second Class    Dv-13045   

     customer_name    segment        country         city  ...  \
0      Claire Gute   Consumer  United States    Henderson  ...   
1      Claire Gute   Consumer  United States    Henderson  ...   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ...   

          category sub-category  \
0        Furniture    Bookcases   
1        Furniture       Chairs   
2  Office Supplies       Labels   

                                        product_name   sales quantity  \
0                  Bush So